# MapReduce simulado em Python

Este notebook simula o MapReduce sem Hadoop nem Spark: a entrada é dividida em blocos, cada bloco passa pela função *map*, os pares intermediários são embaralhados por chave entre os redutores e cada redutor aplica a função *reduce*. As fases ficam escritas por extenso, em funções separadas, para que o caminho de cada par seja visível.

Os exemplos seguem a nota sobre MapReduce: chamadas por número de origem, o efeito do combinador sobre o volume embaralhado, a média que não se combina diretamente e a contagem de distintos por DDD com HyperLogLog como combinador e redutor.

In [1]:
import hashlib
import math
import random
import zlib

random.seed(3)

## Dados: o registro de chamadas

Cada linha é uma tupla (origem, destino, instante, duração). O registro tem 400.000 chamadas, divididas em 8 blocos de 50.000 — no sistema real, os blocos do arquivo no sistema de arquivos distribuído, cada um processado por um mapeador.

In [2]:
ddds = [11, 12, 13, 19, 21, 27, 31, 41, 47, 48, 51, 61, 62, 71, 81, 85, 91, 92]

def numero_aleatorio():
    ddd = random.choice(ddds)
    assinante = random.randint(0, 99_999_999)
    return f"+55{ddd}9{assinante:08d}"

assinantes = []
for i in range(30_000):
    assinantes.append(numero_aleatorio())

registro = []
for i in range(400_000):
    origem = random.choice(assinantes)
    destino = random.choice(assinantes)
    instante = 1_762_000_000 + i
    duracao = round(random.expovariate(1 / 90), 1)
    registro.append((origem, destino, instante, duracao))

tamanho_bloco = 50_000
blocos = []
for inicio in range(0, len(registro), tamanho_bloco):
    blocos.append(registro[inicio:inicio + tamanho_bloco])
print(len(registro), "chamadas em", len(blocos), "blocos")
print("primeira linha:", registro[0])

400000 chamadas em 8 blocos
primeira linha: ('+5581950784412', '+5585954633925', 1762000000, 26.3)


## As três fases

O particionamento usa `zlib.crc32`, e não o `hash()` do Python, porque o `hash()` de cadeias de caracteres muda a cada execução do interpretador; o destino de cada chave precisa ser o mesmo em todas as máquinas.

In [3]:
R = 4   # número de redutores

def fase_map(bloco):
    pares = []
    for linha in bloco:
        origem, destino, instante, duracao = linha
        pares.append((origem, 1))
    return pares

def embaralhar(saidas_dos_mapeadores, R):
    particoes = []
    for r in range(R):
        particoes.append({})
    pares_transportados = 0
    for pares in saidas_dos_mapeadores:
        for chave, valor in pares:
            r = zlib.crc32(chave.encode()) % R     # função de particionamento
            if chave not in particoes[r]:
                particoes[r][chave] = []
            particoes[r][chave].append(valor)
            pares_transportados = pares_transportados + 1
    return particoes, pares_transportados

def fase_reduce(particao):
    resultado = {}
    for chave, valores in particao.items():
        total = 0
        for v in valores:
            total = total + v
        resultado[chave] = total
    return resultado

In [4]:
saidas = []
for bloco in blocos:
    saidas.append(fase_map(bloco))

particoes, sem_combinador = embaralhar(saidas, R)

chamadas_por_origem = {}
for particao in particoes:
    parcial = fase_reduce(particao)
    for chave, total in parcial.items():
        chamadas_por_origem[chave] = total

print("pares embaralhados:", sem_combinador)
for r in range(R):
    print(f"redutor {r}: {len(particoes[r])} chaves")

pares embaralhados: 400000
redutor 0: 7460 chaves
redutor 1: 7502 chaves
redutor 2: 7590 chaves
redutor 3: 7448 chaves


O resultado é conferido contra a contagem direta, que seria o `SELECT origem, COUNT(*) ... GROUP BY origem` numa máquina só.

In [5]:
contagem_direta = {}
for origem, destino, instante, duracao in registro:
    if origem not in contagem_direta:
        contagem_direta[origem] = 0
    contagem_direta[origem] = contagem_direta[origem] + 1
print("resultado igual à contagem direta?", chamadas_por_origem == contagem_direta)

resultado igual à contagem direta? True


## O combinador

O combinador soma as contagens dentro de cada mapeador antes do embaralhamento. Como a soma é associativa e comutativa, o resultado final não muda; muda o número de pares que cruzam a rede.

In [6]:
def combinar(pares):
    acumulado = {}
    for chave, valor in pares:
        if chave not in acumulado:
            acumulado[chave] = 0
        acumulado[chave] = acumulado[chave] + valor
    combinados = []
    for chave, valor in acumulado.items():
        combinados.append((chave, valor))
    return combinados

saidas_combinadas = []
for bloco in blocos:
    saidas_combinadas.append(combinar(fase_map(bloco)))

particoes, com_combinador = embaralhar(saidas_combinadas, R)
resultado_combinado = {}
for particao in particoes:
    parcial = fase_reduce(particao)
    for chave, total in parcial.items():
        resultado_combinado[chave] = total

print("pares embaralhados sem combinador:", sem_combinador)
print("pares embaralhados com combinador:", com_combinador)
print("redução:", round(sem_combinador / com_combinador, 1), "vezes")
print("resultado igual?", resultado_combinado == contagem_direta)

pares embaralhados sem combinador: 400000
pares embaralhados com combinador: 194733
redução: 2.1 vezes
resultado igual? True


A redução é modesta aqui porque cada número aparece poucas vezes por bloco (30.000 assinantes para 50.000 chamadas por bloco). Na contagem de palavras, em que palavras como "de" aparecem milhares de vezes por bloco, a redução é de ordens de grandeza.

## A média não se combina diretamente

Duração média das chamadas por DDD de origem. O combinador ingênuo envia a média de cada mapeador, e o redutor tira a média das médias.

In [7]:
def map_duracao(bloco):
    pares = []
    for origem, destino, instante, duracao in bloco:
        pares.append((origem[3:5], duracao))
    return pares

Com blocos de mesmo tamanho e DDDs distribuídos por igual, a média das médias ficaria próxima da média verdadeira, e o erro passaria despercebido. O erro aparece quando os grupos têm tamanhos diferentes — o caso normal, em que um bloco tem muito mais chamadas de um DDD que outro. Para mostrá-lo, o último bloco é reduzido a 2.000 chamadas, todas 600 s mais longas.

In [8]:
bloco_pequeno = []
for i in range(2_000):
    origem, destino, instante, duracao = blocos[-1][i]
    bloco_pequeno.append((origem, destino, instante, duracao + 600))

blocos_desiguais = blocos[:-1] + [bloco_pequeno]
registro_desigual = []
for bloco in blocos_desiguais:
    for linha in bloco:
        registro_desigual.append(linha)

ddd_exemplo = "11"

# Exato
soma = 0.0
contagem = 0
for origem, destino, instante, duracao in registro_desigual:
    if origem[3:5] == ddd_exemplo:
        soma = soma + duracao
        contagem = contagem + 1
media_exata = soma / contagem

# Média das médias locais
medias = []
somas_e_contagens = []
for bloco in blocos_desiguais:
    soma_local = 0.0
    contagem_local = 0
    for ddd, duracao in map_duracao(bloco):
        if ddd == ddd_exemplo:
            soma_local = soma_local + duracao
            contagem_local = contagem_local + 1
    medias.append(soma_local / contagem_local)
    somas_e_contagens.append((soma_local, contagem_local))

media_das_medias = sum(medias) / len(medias)

soma_total = 0.0
contagem_total = 0
for soma_local, contagem_local in somas_e_contagens:
    soma_total = soma_total + soma_local
    contagem_total = contagem_total + contagem_local
media_pelo_par = soma_total / contagem_total

print(f"DDD {ddd_exemplo}: média exata {media_exata:.2f} s")
print(f"média das médias locais:  {media_das_medias:.2f} s   (errada)")
print(f"par (soma, contagem):     {media_pelo_par:.2f} s   (igual à exata)")

DDD 11: média exata 94.45 s
média das médias locais:  164.95 s   (errada)
par (soma, contagem):     94.45 s   (igual à exata)


## Contagem de distintos por DDD com HyperLogLog

Quantos números distintos ligaram para cada DDD de destino. Contagens de distintos não se somam entre mapeadores; o *sketch* HyperLogLog se funde pelo máximo registrador a registrador, operação associativa e comutativa, e por isso serve como combinador e como redutor.

O HyperLogLog é o mesmo do notebook de HyperLogLog e MinHash, repetido aqui por inteiro.

In [9]:
p = 12
m = 2 ** p

def hll_novo():
    return [0] * m

def hll_inserir(registradores, item):
    h = int.from_bytes(hashlib.blake2b(item.encode(), digest_size=8).digest(), "big")
    j = h >> (64 - p)
    resto = h & ((1 << (64 - p)) - 1)
    rho = (64 - p) - resto.bit_length() + 1
    if rho > registradores[j]:
        registradores[j] = rho

def hll_fundir(a, b):
    fundido = []
    for j in range(m):
        fundido.append(max(a[j], b[j]))
    return fundido

def hll_estimar(registradores):
    alfa = 0.7213 / (1 + 1.079 / m)
    soma = 0.0
    zerados = 0
    for valor in registradores:
        soma = soma + 2.0 ** (-valor)
        if valor == 0:
            zerados = zerados + 1
    estimativa = alfa * m * m / soma
    if estimativa < 2.5 * m and zerados > 0:
        estimativa = m * math.log(m / zerados)
    return estimativa

print("registradores:", m, " erro padrão previsto:", round(1.04 / math.sqrt(m) * 100, 2), "%")

registradores: 4096  erro padrão previsto: 1.62 %


O mapeador mantém um *sketch* por DDD ao longo do bloco e emite todos no fim — o combinador já embutido no mapeador. O embaralhamento transporta, no máximo, um *sketch* por DDD por mapeador.

In [10]:
def map_distintos(bloco):
    sketches = {}
    for origem, destino, instante, duracao in bloco:
        ddd = destino[3:5]
        if ddd not in sketches:
            sketches[ddd] = hll_novo()
        hll_inserir(sketches[ddd], origem)
    pares = []
    for ddd, sketch in sketches.items():
        pares.append((ddd, sketch))
    return pares

def reduce_distintos(particao):
    resultado = {}
    for ddd, sketches in particao.items():
        fundido = hll_novo()
        for sketch in sketches:
            fundido = hll_fundir(fundido, sketch)
        resultado[ddd] = hll_estimar(fundido)
    return resultado

saidas = []
for bloco in blocos:
    saidas.append(map_distintos(bloco))
particoes, transportados = embaralhar(saidas, R)

estimado_por_ddd = {}
for particao in particoes:
    parcial = reduce_distintos(particao)
    for ddd, estimativa in parcial.items():
        estimado_por_ddd[ddd] = estimativa

print("sketches embaralhados:", transportados, "(no lugar de", len(registro), "pares (DDD, origem))")

sketches embaralhados: 144 (no lugar de 400000 pares (DDD, origem))


In [11]:
exato_por_ddd = {}
for origem, destino, instante, duracao in registro:
    ddd = destino[3:5]
    if ddd not in exato_por_ddd:
        exato_por_ddd[ddd] = set()
    exato_por_ddd[ddd].add(origem)

soma_das_contagens_locais = {}
for bloco in blocos:
    locais = {}
    for origem, destino, instante, duracao in bloco:
        ddd = destino[3:5]
        if ddd not in locais:
            locais[ddd] = set()
        locais[ddd].add(origem)
    for ddd, conjunto in locais.items():
        if ddd not in soma_das_contagens_locais:
            soma_das_contagens_locais[ddd] = 0
        soma_das_contagens_locais[ddd] = soma_das_contagens_locais[ddd] + len(conjunto)

print(f"{'DDD':>4} {'exato':>7} {'HLL':>7} {'erro':>7} {'soma das contagens locais':>27}")
for ddd in ["11", "21", "31", "61", "91"]:
    exato = len(exato_por_ddd[ddd])
    estimado = estimado_por_ddd[ddd]
    print(f"{ddd:>4} {exato:>7} {estimado:>7.0f} {(estimado - exato) / exato:>7.1%} {soma_das_contagens_locais[ddd]:>27}")

 DDD   exato     HLL    erro   soma das contagens locais
  11   15603   15822    1.4%                       21149
  21   15597   15922    2.1%                       21052
  31   15433   15621    1.2%                       20775
  61   16001   16184    1.1%                       21737
  91   15908   15995    0.5%                       21535


A última coluna é o que um combinador ingênuo produziria somando as contagens de distintos de cada bloco: os números que ligaram para o mesmo DDD em blocos diferentes são contados várias vezes. A fusão dos *sketches* não tem esse defeito, e o erro fica dentro do erro padrão de um único HyperLogLog com $m = 4.096$ registradores.